# TopoMedVision — exploration notebook

End-to-end walk-through of the persistent-homology pipeline on one synthetic MRI slice. Run from the project root so the `backend` import resolves.

Cells:
1. Load + preprocess a sample slice
2. Compute cubical persistence (dim 0 + dim 1)
3. Visualize the diagram and barcode
4. Build a topology-derived highlight mask
5. Compute the feature vector and score

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # repo root

from backend import utils, persistence, hybrid_model, visualization
import matplotlib.pyplot as plt

img = utils.load_image('../data/samples/05_tumor_ring.png')
img = utils.preprocess(img)
plt.imshow(img, cmap='gray'); plt.axis('off'); plt.show()

In [ ]:
result = persistence.compute_cubical_persistence(img)
print('backend:', result['backend'])
print('Betti numbers:', result['betti'])
print('top 5 dim0 by persistence:')
for p in result['dim0'][:5]:
    print(f'  birth={p.birth:.3f} death={p.death:.3f} pers={p.persistence:.3f} xy={p.birth_xy}')

In [ ]:
fig = visualization.persistence_diagram(result['dim0'], result['dim1'])
plt.show()
fig = visualization.persistence_barcode(result['dim0'], result['dim1'])
plt.show()

In [ ]:
pairs = list(result['dim0']) + list(result['dim1'])
mask = persistence.topology_mask(img, pairs, top_k=5)
overlay = utils.overlay_mask(img, mask)
plt.imshow(overlay); plt.axis('off'); plt.show()

In [ ]:
score = hybrid_model.score_tumor_likelihood(result['dim0'], result['dim1'], image=img)
print(f'score = {score.score:.3f}')
print(score.explanation)